In [1]:
!pip install -q crewai crewai-tools deepeval groq langchain langchain-core langchain-groq langchain-community faiss-cpu sentence-transformers langchain-huggingface litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.2/784.2 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from crewai import Agent, Task, Crew
from crewai.tools import tool
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from groq import Groq
import getpass
import json
import os
import re
import time
import pandas as pd

## Part 1: Knowledge Base

### Knowledge Base: Framework Laptop 13 DIY Edition Quick Start Guide

The Framework Laptop 13 DIY Edition Quick Start Guide provides a structured, step-by-step process for assembling and setting up a modular laptop designed for repairability and customization. The guide is intended for users who receive a partially assembled laptop and must complete the installation of key components such as memory, storage, and expansion modules before powering on the device.

A central feature of the Framework Laptop is its DIY assembly model, where users actively participate in configuring their system. Unlike traditional laptops that arrive fully sealed, the Framework Laptop allows users to install or upgrade hardware components themselves. This includes inserting RAM, SSD storage, and expansion cards into designated slots. The guide emphasizes reading each step carefully and reviewing accompanying visuals before proceeding, ensuring correct assembly and minimizing user errors.

The setup process begins with unboxing and verifying components. Users are instructed to confirm that all required items are included, such as the laptop chassis, expansion cards, power adapter, and a specialized screwdriver provided by the manufacturer. These components are essential for completing the assembly process and ensuring that the device can be powered on safely.

A key step in the guide involves the installation of expansion cards, which is one of the defining features of the Framework Laptop. These cards act as modular ports (e.g., USB-C, HDMI, storage modules) and can be inserted into bays on the sides of the laptop. Users can choose which ports they want and arrange them according to their preferences. The cards are inserted by sliding them into the slots until they click into place, sometimes requiring slight force during initial use.

Another important aspect of the setup process is the connection of the power supply. After assembling the necessary components, users connect a USB-C power adapter to one of the expansion card slots. The guide advises users not to power on the device until all assembly steps are completed, ensuring that the system initializes correctly on first boot.

The Framework Laptop's design is centered around modularity and repairability, which distinguishes it from conventional laptops. The internal components are easily accessible, requiring only a few screws to open the device. This allows users to upgrade or replace parts such as RAM, storage, or even the mainboard without specialized tools or professional assistance. This design philosophy supports long-term usability and reduces electronic waste by enabling incremental upgrades instead of full device replacement.

The quick start guide typically includes around 10-20 steps and takes approximately 10-20 minutes to complete, depending on the model and user familiarity. Each step is sequential and designed to guide users from initial unboxing to a fully functional system ready for operating system installation.

After completing the hardware setup, users may proceed to install an operating system, such as Windows or Linux. The guide may reference additional documentation for OS installation and driver setup, indicating that the quick start guide focuses primarily on hardware assembly rather than software configuration.

Overall, the Framework Laptop 13 Quick Start Guide exemplifies a user-centric approach to hardware interaction, combining clear instructions with modular design principles. It enables users to understand and control their device at a deeper level compared to traditional laptops, making it a valuable resource for both beginners and enthusiasts interested in customizable computing systems.

### Key Facts

1. The Framework Laptop 13 DIY Edition requires user assembly of components such as RAM, storage, and expansion cards.
2. The setup process begins with unboxing and verifying included components like the chassis, power adapter, and screwdriver.
3. Expansion cards act as modular ports and can be inserted into bays on the laptop's sides.
4. The laptop should not be powered on until all assembly steps are completed.
5. The device is designed for easy repair and upgrade, allowing access to internal components with minimal tools.
6. The quick start guide typically contains 10-20 steps and takes about 10-20 minutes to complete.
7. Operating system installation is a separate step after hardware assembly.

In [2]:
# Build the Vector Store
loader = TextLoader("/content/manual.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embeddings)
vector_store.save_local("faiss_index")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Choice of topic
#### 1. Simpler & cleaner structure
- The Framework guide is:
- short (≈10-20 minutes process)
- step-by-step (≈13 steps)
- linear (unboxing → assembly → power on)
- chunks are easy to define
- each step is self-contained
- retrieval is very accurate

#### 2. Easier for lightweight LLMs
- The content is procedural
- written in plain English
- supported by clear instructions
- Lightweight models perform much better on instructions

#### 3. Easy evaluation
answers for manual related questions are:
- objective
- easy to verify
<hr>

## Part 2: RAG Agent

In [3]:
GROQ_API_KEY = getpass.getpass("Enter your GROQ API key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

Enter your GROQ API key: ··········


In [32]:
# Define the tool with explicit string return to ensure CrewAI compatibility
@tool("search_manual")
def search_manual_tool(query: str):
    """
    Searches the Framework Laptop manual for specific technical details,
    troubleshooting steps, and hardware specifications.
    """
    results = vector_store.similarity_search(query, k=3)
    content = "\n\n".join([doc.page_content for doc in results])
    return str(content)

In [10]:
# Agent Definition
retriever_agent = Agent(
    role="Framework Laptop Specialist",
    goal="Answer questions strictly using the provided manual context.",
    backstory="Expert in Framework's modular hardware documentation.",
    tools=[search_manual_tool],
    llm = {
        "model": "groq/llama-3.3-70b-versatile",
        "temperature": 0.3,
        "llm_type": "litellm"
    },
    verbose=True,
    allow_delegation=False,
    reasoning=True
)

In [11]:
# Task Definition
rag_task = Task(
    description="""
    Identify the answer for: '{question}'

    You MUST use the tool 'search_manual' EXACTLY like this:
    search_manual({"query": "<your query>"})

    Do NOT modify the tool name.
    Do NOT add extra characters.
    Do NOT call any other tool.

    Strict Rules:
    1. Use ONLY the retrieved context from the search_manual tool.
    2. If the answer is not present, state 'Not found in manual'.
    3. Do not use any internal knowledge about other laptops or general tech.
    """,
    expected_output="""A JSON object containing:
    - 'answer': The specific answer derived from the manual.
    - 'retrieved_context': The exact text snippets used to formulate the answer.""",
    agent=retriever_agent
)

In [15]:
questions = [
    "What items should you check for when unboxing the Framework Laptop?",
    "How do you install expansion cards in the Framework Laptop?",
    "Should you power on the Framework Laptop before completing setup?"
]

crew = Crew(
    agents=[retriever_agent],
    tasks=[rag_task],
    verbose=True
)

for q in questions:
    print(f"Question: {q}")

    # Execute the crew
    result = crew.kickoff(inputs={"question": q})

    # Handle CrewAI TaskOutput object or string
    raw_output = str(result.raw if hasattr(result, 'raw') else result)

    try:
        # Clean the output in case the LLM wrapped it in markdown code blocks
        clean_json = re.sub(r'^```json\s*|\s*```$', '', raw_output.strip(), flags=re.MULTILINE)
        parsed = json.loads(clean_json)

        print(f"\nQ: {q}")
        print(f"A: {parsed.get('answer', 'No answer key found')}")
        print(f"Context:\n{parsed.get('retrieved_context', 'No context key found')}")
    except Exception as e:
        print(f"\nParsing Error: {e}")
        print("Raw Output:", raw_output)

    print("-" * 30)
    time.sleep(5)
    print()

Question: What items should you check for when unboxing the Framework Laptop?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d47e50c3-aaa8-40b7-9086-4e7ed70b526f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Identify the answer for: 'What items should you check for when unboxing the Framework Laptop?'             │
│                                                                                                                 │
│      You MUST use the tool 'search_manual' EXACTLY like this:                                                   │
│      search_manual({"query": "<your query>"})                                                                   │
│                                                                                                                 │
│      Do NOT modify the tool name.                                                                               │
│      Do NOT add extra characters.                                                                               │
│      Do NOT call any other tool.                                                                                │
│                                                                                                                 │
│      Strict Rules:                                                                                              │
│      1. Use ONLY the retrieved context from the search_manual tool.                                             │
│      2. If the answer is not present, state 'Not found in manual'.                                              │
│      3. Do not use any internal knowledge about other laptops or general tech.                                  │
│                                                                                                                 │
│  ID: abbc7b52-d9bf-4361-ae90-593ddbba730d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🧠 Reasoning ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Reasoning Started                                                                                              │
│  Attempt: 1                                                                                                     │
│  Status: Thinking...                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🔧 LLM Tool Usage ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Started                                                                                             │
│  Name: create_reasoning_plan                                                                                    │
│  Status: In Progress                                                                                            │
│  Tool Args: {'plan': 'Unboxing Framework Laptop Checklist', 'ready': True, 'steps': [{'depends_on': [],         │
│  'description': "Search the manual for unboxing checklist using the query 'unboxing checklist'",                │
│  'step_number': 1, 'tool_to_use': 'search_manual'}, {'depends_on': [1], 'description': 'Extract relevant text   │
│  snippets from the search results', 'step_number': 2, 'tool_to_use': None}, {'depends_on': [2], 'description':  │
│  'Formulate the answer based on the extracted text snippets', 'step_number': 3, 'tool_to_use': None},           │
│  {'depends_on': [3], 'description': "Create the output JSON object with 'answer' and 'retrieved_context'",      │
│  'step_number': 4, 'tool_to_use': None}]}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for     │
│  model `llama-3.3-70b-versatile` in organization `org_01jjrr2sw9e2ctpzbbv802je4t` service tier `on_demand` on   │
│  tokens per minute (TPM): Limit 12000, Used 8619, Requested 4950. Please try again in 7.845s. Need more         │
│  tokens? Upgrade to Dev Tier today at                                                                           │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│      Identify the answer for: 'How do you install expansion cards in the Framework Laptop?'                     │
│                                                                                                                 │
│      You MUST use the tool 'search_manual' EXACTLY like this:                                                   │
│      search_manual({"query": "<your query>"})                                                                   │
│                                                                                                                 │
│      Do NOT modify the tool name.                                                                               │
│      Do NOT add extra characters.                                                                               │
│      Do NOT call any other tool.                                                                                │
│                                                                                                                 │
│      Strict Rules:                                                                                              │
│      1. Use ONLY the retrieved context from the search_manual tool.                                             │
│      2. If the answer is not present, state 'Not found in manual'.                                              │
│      3. Do not use any internal knowledge about other laptops or general tech.                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Planning:                                                                                                      │
│  Install expansion cards in Framework Laptop                                                                    │
│  Agent: Framework Laptop Specialist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: d47e50c3-aaa8-40b7-9086-4e7ed70b526f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjrr2sw9e2ctpzbbv802je4t` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 8619, Requested 4950. Please try again in 7.845s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}




╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

## Part 3: Quality Evaluator Agent

In [19]:
@tool("evaluate_rag_quality")
def evaluation_tool(question: str, answer: str, context: str):
    """
    Evaluates the RAG system's output for faithfulness and relevancy.
    Threshold for success is 0.7.
    """
    # Create the Test Case
    # DeepEval expects retrieval_context as a list of strings
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[context]
    )

    # Initialize Metrics (Threshold = 0.7)
    f_metric = FaithfulnessMetric(threshold=0.7, model="groq/llama-3.3-70b-versatile")
    r_metric = AnswerRelevancyMetric(threshold=0.7, model="groq/llama-3.3-70b-versatile")

    # Measure
    f_metric.measure(test_case)
    r_metric.measure(test_case)

    # Determine Verdict
    verdict = "PASS" if (f_metric.score >= 0.7 and r_metric.score >= 0.7) else "FAIL"

    # Build structured results
    results = {
        "faithfulness_score": round(f_metric.score, 2),
        "relevancy_score": round(r_metric.score, 2),
        "verdict": verdict,
        "reasons": {
            "faithfulness_reason": f_metric.reason,
            "relevancy_reason": r_metric.reason
        }
    }

    return json.dumps(results, indent=2)

In [20]:
# Agent Definition
evaluator_agent = Agent(
    role="Quality Assurance Specialist",
    goal="Objectively evaluate the quality of RAG answers based on faithfulness and relevancy.",
    backstory="""You are a rigorous auditor. Your job is to verify that the support team
    isn't hallucinating and is actually helping the customer with the Framework Laptop.""",
    tools=[evaluation_tool],
    llm = {
        "model": "groq/llama-3.3-70b-versatile",
        "temperature": 0.3,
        "llm_type": "litellm"
    },
    verbose=True
)

In [21]:
# Task Definition
eval_task = Task(
    description="""
    Review the output from the RAG Specialist.

    1. Extract the 'question', 'answer', and 'retrieved_context' from the previous task.
    2. Run the evaluate_rag_quality tool.
    3. If the verdict is FAIL, you must be extremely specific about why in the reasons section.
    """,
    expected_output="""A JSON verdict including scores for faithfulness and relevancy,
    the PASS/FAIL status, and detailed reasons for the scores.""",
    agent=evaluator_agent,
    context=[rag_task]
)

In [22]:
# Assemble the crew with the tasks defined so far
evaluation_crew = Crew(
    agents=[retriever_agent, evaluator_agent],
    tasks=[rag_task, eval_task],
    verbose=True
)

# Run the pipeline
# The 'question' input will be passed into the rag_task description
results = evaluation_crew.kickoff(inputs={"question": "How do I upgrade the RAM?"})

print(results)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 769b1343-d7f2-4fdf-84bf-2e832131d58c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🧠 Reasoning ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Reasoning Started                                                                                              │
│  Attempt: 1                                                                                                     │
│  Status: Thinking...                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Identify the answer for: 'How do I upgrade the RAM?'                                                       │
│                                                                                                                 │
│      You MUST use the tool 'search_manual' EXACTLY like this:                                                   │
│      search_manual({"query": "<your query>"})                                                                   │
│                                                                                                                 │
│      Do NOT modify the tool name.                                                                               │
│      Do NOT add extra characters.                                                                               │
│      Do NOT call any other tool.                                                                                │
│                                                                                                                 │
│      Strict Rules:                                                                                              │
│      1. Use ONLY the retrieved context from the search_manual tool.                                             │
│      2. If the answer is not present, state 'Not found in manual'.                                              │
│      3. Do not use any internal knowledge about other laptops or general tech.                                  │
│                                                                                                                 │
│  ID: abbc7b52-d9bf-4361-ae90-593ddbba730d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🔧 LLM Tool Usage ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Started                                                                                             │
│  Name: create_reasoning_plan                                                                                    │
│  Status: In Progress                                                                                            │
│  Tool Args: {'plan': "Execute a step-by-step plan to find the answer to the question 'How do I upgrade the      │
│  RAM?' using the search_manual tool", 'ready': True, 'steps': [{'depends_on': [], 'description': "Search the    │
│  manual for 'RAM upgrade'", 'step_number': 1, 'tool_to_use': 'search_manual'}, {'depends_on': [1],              │
│  'description': 'Extract the relevant text snippets from the search results', 'step_number': 2, 'tool_to_use':  │
│  None}, {'depends_on': [2], 'description': 'Formulate the answer based on the extracted text snippets',         │
│  'step_number': 3, 'tool_to_use': None}, {'depends_on': [3], 'description': 'Create the output JSON object      │
│  with the answer and retrieved context', 'step_number': 4, 'tool_to_use': None}]}                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ LLM Tool Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Completed                                                                                           │
│  Tool: create_reasoning_plan                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│      Review the output from the RAG Specialist.                                                                 │
│                                                                                                                 │
│      1. Extract the 'question', 'answer', and 'retrieved_context' from the previous task.                       │
│      2. Run the evaluate_rag_quality tool.                                                                      │
│      3. If the verdict is FAIL, you must be extremely specific about why in the reasons section.                │
│                                                                                                                 │
│  Agent: Quality Assurance Specialist                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 769b1343-d7f2-4fdf-84bf-2e832131d58c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjrr2sw9e2ctpzbbv802je4t` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11551, Requested 774. Please try again in 1.625s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


## Part 4: Revisor Agent

In [23]:
# Agent Definition
revisor_agent = Agent(
    role="Technical Editor",
    goal="Correct and refine answers that fail quality standards.",
    backstory="""You are a perfectionist editor. If the Quality Specialist finds
    a hallucination or an irrelevant answer, you rewrite it using ONLY the
    provided context to ensure 100% accuracy.""",
    llm = {
        "model": "groq/llama-3.3-70b-versatile",
        "temperature": 0.3,
        "llm_type": "litellm"
    },
    verbose=True
)

In [24]:
# Task Definition
revision_task = Task(
    description="""
    1. Check the verdict from the Quality Auditor.
    2. If the verdict is 'PASS', repeat the original answer exactly.
    3. If the verdict is 'FAIL', rewrite the answer for the question: '{question}'.
    4. Use the 'reasons' provided by the auditor and the original 'retrieved_context'
       to fix all inaccuracies or missing information.
    """,
    expected_output="A finalized, high-quality answer grounded in the manual.",
    agent=revisor_agent,
    context=[eval_task, rag_task] # Accesses both the evaluation and the original RAG output
)

In [25]:
# Assemble the full pipeline
full_crew = Crew(
    agents=[retriever_agent, evaluator_agent, revisor_agent],
    tasks=[rag_task, eval_task, revision_task],
    verbose=True
)

# Run an adversarial test to trigger the Revisor
adversarial_input = {"question": "How do I water-cool the Framework laptop CPU?"}
final_result = full_crew.kickoff(inputs=adversarial_input)

revised_answer = str(final_result)
context_used = rag_task.output.json_dict['retrieved_context']

# Re-run evaluation on the revised answer
final_score = evaluation_tool.fn(
    question=adversarial_input["question"],
    answer=revised_answer,
    context=context_used
)

print(f"Revised Answer: {revised_answer}")
print(f"Re-scored Metrics: {final_score}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 744b31b8-e672-473a-bc39-a209190d3f44                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Identify the answer for: 'How do I water-cool the Framework laptop CPU?'                                   │
│                                                                                                                 │
│      You MUST use the tool 'search_manual' EXACTLY like this:                                                   │
│      search_manual({"query": "<your query>"})                                                                   │
│                                                                                                                 │
│      Do NOT modify the tool name.                                                                               │
│      Do NOT add extra characters.                                                                               │
│      Do NOT call any other tool.                                                                                │
│                                                                                                                 │
│      Strict Rules:                                                                                              │
│      1. Use ONLY the retrieved context from the search_manual tool.                                             │
│      2. If the answer is not present, state 'Not found in manual'.                                              │
│      3. Do not use any internal knowledge about other laptops or general tech.                                  │
│                                                                                                                 │
│  ID: abbc7b52-d9bf-4361-ae90-593ddbba730d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🧠 Reasoning ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Reasoning Started                                                                                              │
│  Attempt: 1                                                                                                     │
│  Status: Thinking...                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🔧 LLM Tool Usage ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Started                                                                                             │
│  Name: create_reasoning_plan                                                                                    │
│  Status: In Progress                                                                                            │
│  Tool Args: {'plan': 'Search manual for water-cooling Framework laptop CPU', 'ready': True, 'steps':            │
│  [{'depends_on': [], 'description': 'Search manual for water-cooling Framework laptop CPU', 'step_number': 1,   │
│  'tool_to_use': 'search_manual'}, {'depends_on': [1], 'description': 'Extract relevant text snippets from       │
│  search results', 'step_number': 2, 'tool_to_use': None}, {'depends_on': [2], 'description': 'Formulate answer  │
│  based on extracted text snippets', 'step_number': 3, 'tool_to_use': None}, {'depends_on': [3], 'description':  │
│  'Create output JSON object with answer and retrieved context', 'step_number': 4, 'tool_to_use': None}]}        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ LLM Tool Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Completed                                                                                           │
│  Tool: create_reasoning_plan                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│      Identify the answer for: 'How do I water-cool the Framework laptop CPU?'                                   │
│                                                                                                                 │
│      You MUST use the tool 'search_manual' EXACTLY like this:                                                   │
│      search_manual({"query": "<your query>"})                                                                   │
│                                                                                                                 │
│      Do NOT modify the tool name.                                                                               │
│      Do NOT add extra characters.                                                                               │
│      Do NOT call any other tool.                                                                                │
│                                                                                                                 │
│      Strict Rules:                                                                                              │
│      1. Use ONLY the retrieved context from the search_manual tool.                                             │
│      2. If the answer is not present, state 'Not found in manual'.                                              │
│      3. Do not use any internal knowledge about other laptops or general tech.                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Planning:                                                                                                      │
│  Search manual for water-cooling Framework laptop CPU                                                           │
│  Agent: Framework Laptop Specialist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 744b31b8-e672-473a-bc39-a209190d3f44                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjrr2sw9e2ctpzbbv802je4t` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 6230, Requested 6163. Please try again in 1.965s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Part 5: Full pipeline

In [27]:
test_questions = [
    # 5 Questions from the manual
    "How do I install the Expansion Cards?",
    "Should the laptop be powered on during assembly?",
    "Can I upgrade the device?",
    "How long is the quick start process?",
    "Can I install Linux on this laptop?",

    # 2 Adversarial Questions
    "How do I change the oil in my Framework laptop?",
    "What is the best recipe for baking a cake using the laptop's heat?"
]

In [33]:
# Results storage
results_data = []

# Initialize the Full Crew
full_crew = Crew(
    agents=[retriever_agent, evaluator_agent, revisor_agent],
    tasks=[rag_task, eval_task, revision_task],
    verbose=False
)

for q in test_questions:
    print(f"Processing: {q}")

    try:
        # 1. Run the full pipeline
        crew_result = full_crew.kickoff(inputs={"question": q})

        # 2. Extract Initial Scores safely
        eval_raw = getattr(eval_task.output, 'raw', "{}")
        eval_raw_clean = re.sub(r'^```json\s*|\s*```$', '', str(eval_raw).strip(), flags=re.MULTILINE)

        try:
            eval_output = json.loads(eval_raw_clean)
        except:
            eval_output = {}

        initial_f = eval_output.get("faithfulness_score", 0)
        initial_r = eval_output.get("relevancy_score", 0)
        verdict = eval_output.get("verdict", "FAIL")

        # 3. Get Final Scores
        final_answer = str(crew_result)
        rag_out_dict = getattr(rag_task.output, 'json_dict', {}) or {}
        context_used = rag_out_dict.get('retrieved_context', "Context not captured")

        final_eval_json = evaluation_tool.fn(question=q, answer=final_answer, context=str(context_used))
        final_eval = json.loads(final_eval_json)

        results_data.append({
            "Question": q,
            "Initial Faithfulness": initial_f,
            "Initial Relevancy": initial_r,
            "Verdict": verdict,
            "Final Faithfulness": final_eval.get("faithfulness_score", 0),
            "Final Relevancy": final_eval.get("relevancy_score", 0)
        })
    except Exception as e:
        print(f"Error processing '{q}': {e}")

    time.sleep(8)

# Create DataFrame
df_results = pd.DataFrame(results_data)
display(df_results) if not df_results.empty else print("No results to display.")

Processing: How do I install the Expansion Cards?
Maximum iterations reached. Requesting final answer.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Framework Laptop Specialist                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Identify the answer for: 'How do I install the Expansion Cards?'                                           │
│                                                                                                                 │
│      You MUST use the tool 'search_manual' EXACTLY like this:                                                   │
│      search_manual({"query": "<your query>"})                                                                   │
│                                                                                                                 │
│      Do NOT modify the tool name.                                                                               │
│      Do NOT add extra characters.                                                                               │
│      Do NOT call any other tool.                                                                                │
│                                                                                                                 │
│      Strict Rules:                                                                                              │
│      1. Use ONLY the retrieved context from the search_manual tool.                                             │
│      2. If the answer is not present, state 'Not found in manual'.                                              │
│      3. Do not use any internal knowledge about other laptops or general tech.                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Planning:                                                                                                      │
│  Install Expansion Cards Plan                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Framework Laptop Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  {"answer": "Expansion cards are installed by sliding them into the slots until they click into place,          │
│  sometimes requiring slight force during initial use.", "retrieved_context": "A key step in the guide involves  │
│  the installation of expansion cards, which is one of the defining features of the Framework Laptop. These      │
│  cards act as modular ports (e.g., USB-C, HDMI, storage modules) and can be inserted into bays on the sides of  │
│  the laptop. Users can choose which ports they want and arrange them according to their preferences. The cards  │
│  are inserted by sliding them into the slots until they click into place, sometimes requiring slight force      │
│  during initial use."}                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quality Assurance Specialist                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Review the output from the RAG Specialist.                                                                 │
│                                                                                                                 │
│      1. Extract the 'question', 'answer', and 'retrieved_context' from the previous task.                       │
│      2. Run the evaluate_rag_quality tool.                                                                      │
│      3. If the verdict is FAIL, you must be extremely specific about why in the reasons section.                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool evaluate_rag_quality executed with result: Error executing tool: OpenAI API key is not configured. Set OPENAI_API_KEY in your environment or pass `api_key` to GPTModel(...)....


...

Processing: What is the best recipe for baking a cake using the laptop's heat?
Maximum iterations reached. Requesting final answer.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Framework Laptop Specialist                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Identify the answer for: 'What is the best recipe for baking a cake using the laptop's heat?'              │
│                                                                                                                 │
│      You MUST use the tool 'search_manual' EXACTLY like this:                                                   │
│      search_manual({"query": "<your query>"})                                                                   │
│                                                                                                                 │
│      Do NOT modify the tool name.                                                                               │
│      Do NOT add extra characters.                                                                               │
│      Do NOT call any other tool.                                                                                │
│                                                                                                                 │
│      Strict Rules:                                                                                              │
│      1. Use ONLY the retrieved context from the search_manual tool.                                             │
│      2. If the answer is not present, state 'Not found in manual'.                                              │
│      3. Do not use any internal knowledge about other laptops or general tech.                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Planning:                                                                                                      │
│  Execute a series of steps to find the best recipe for baking a cake using the laptop's heat by searching the   │
│  manual                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Error processing 'What is the best recipe for baking a cake using the laptop's heat?': litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jjrr2sw9e2ctpzbbv802je4t` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99161, Requested 11152. Please try again in 2h28m30.432s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

No results to display.
